## Introduction
This notebook will perform extensive data analysis regarding bird bioacoustic monitoring. The data will be taken from the article "Automated identification of individual birds by song enables multi-year recapture from passive acoustic monitoring data", and it is located under resources/localized_events.csv. 

In [ ]:
# First, we must import the necessary packages and libraries
import pandas as pd
import ast
from pathlib import Path 
from plotnine import ( 
    ggplot, 
    aes, 
    geom_point,
    geom_bar,
    facet_wrap,
    labs,
    scale_fill_manual,
    theme_minimal
)

Then, we will need to load in the data as a dataframe or just fetch the binary if it exists

In [ ]:
path = Path("resources/localized_events.parquet")

if path.exists():
    print("Exists, so just load in binary file")
    all_loaded_events_df = pd.read_parquet("resources/localized_events.parquet")
else:
    print("Does not exist, so we create it")
    all_loaded_events_df = pd.read_csv("resources/localized_events.csv")
    all_loaded_events_df.to_parquet("resources/localized_events.parquet")


Now, we will take a subset of the column variables in the data spreadsheet.

In [ ]:
subst_loaded_events_df = all_loaded_events_df[
    [
    # Metadata
    'array',
    'data_split',
    'relative_position',
    'mic_distances_m',
    'start_timestamp',
    'duration',

    # Automated models
    'ovenbird_cluster_labels',
    'baseline_resnet18_cluster_labels',
    'hawkears_cluster_labels',
    'birdnet_cluster_labels',
    ]
]

### Exploratory Data Analysis
We will begin by exploring the distributions of the data to identify possible outliers, trends, and occurrences. 

In [ ]:
# add a new column individual so it does not conflict with python
subst_loaded_events_df['individual'] = all_loaded_events_df['class']

# Convert to datetime objects
subst_loaded_events_df['start_timestamp'] = pd.to_datetime(subst_loaded_events_df['start_timestamp'], utc=True)
subst_loaded_events_df['duration'] = pd.to_timedelta(subst_loaded_events_df['duration'], unit='minute')

In [ ]:
# Create a new column just for the end_timestamp -> datetime object
subst_loaded_events_df['end_timestamp'] = (subst_loaded_events_df['start_timestamp'] + subst_loaded_events_df['duration'])

In [ ]:
# Set an event key for all the rows
subst_loaded_events_df['event_key'] = range(len(subst_loaded_events_df))

In [ ]:
# .value_counts() creates a lookup table, (individual: counts) for all individuals
# .map() takes that lookup table and assigns it to those individuals using 'individual' as the key
# This new column 'count' is assigned with that counts for each individual
lookup_individual_counts = subst_loaded_events_df['individual'].value_counts()
subst_loaded_events_df['count'] = subst_loaded_events_df['individual'].map(
    lookup_individual_counts
)

In [ ]:
# Save to under resources
subst_loaded_events_df.to_parquet('resources/subset_loaded_events.parquet')
subst_loaded_events_df.to_csv('resources/subset_loaded_events.csv')

In [ ]:
# Creates a ggplot object 
(
    ggplot(subst_loaded_events_df, aes(x='factor(individual)', fill='count > 30'))
    + geom_bar()
    + scale_fill_manual(
        values={True: 'blue', False: 'red'}
    )
    + labs(
        title='Song Count per Individual',
        x='Individual',
        y='Count'
    )
)

Now, we want to plot the recordings according to the relative (x, y) coordinates, they will be labeled according to the individual they belong to & faceted by the sites they belong to.

In [ ]:
# Create the figure for this, using ast package to unpack the str
fig_df = pd.DataFrame()
fig_df['x'] = subst_loaded_events_df['relative_position'].apply(
    lambda p: ast.literal_eval(p)[0]
)
fig_df['y'] = subst_loaded_events_df['relative_position'].apply(
    lambda p: ast.literal_eval(p)[1]
)
fig_df['individual'] = subst_loaded_events_df['individual'].astype("category")

In [ ]:
(
    ggplot(fig_df, aes('x', 'y', fill = 'individual')) +
        geom_point() 
)

Now, we will facet grid them by the site.

In [ ]:
# Introduce the site column to enable facet gridding
fig_df['site'] = subst_loaded_events_df['array']

In [ ]:
(
    ggplot(fig_df, aes('x', 'y', fill = 'individual')) +
        geom_point() +
        facet_wrap('site') +
        labs(
        title='Ovenbird Song Positions by Array',
        x='Easting (m)',
        y='Northing (m)',
        color='Individual'
        ) +
        theme_minimal()
)

## Bird Recordings Overlap
The overlaps of all the bird recordings will be found, and that information will be used to determine bird interactions.
- If two bird recordings overlap, that may or may not be an interaction (further analysis is required)
- If two bird recordings do not overlap, that is not an interaction

### Acoustic Data Processing
First the data must be manipulated and processed prior to running the algorithms.

In [ ]:
from pandas import DataFrame

In [ ]:
def create_tuple_list_from_df(df: DataFrame, start: str, end: str) -> list:
    '''
    From a dataframe of information, return a list of tuples with the index, start, and end timestamps
    '''
    return list(df[[start, end]].itertuples(index=True, name=None))

In [ ]:
# Find the overlap for individual events from a list of all events
def overlap_individual(event, i, total_events, sort: bool) -> list[tuple]:
    '''
    For an individual i, find a list of events that overlap as tuples
    '''
    if not sort:
        total_events = sorted(total_events, key=lambda x: x[1]) # key must use a function for comparison
        
    event_key, _, end = event
    stored_overlaps = []
    for key, start, _ in total_events[i:]:
        if start < end:
            stored_overlaps.append((event_key, key))
        else:
            break
    return stored_overlaps

In [ ]:
# Find the overlap from all the events
def overlap(events) -> list[tuple]:
    '''
    Finds a list of all event overlaps, then returns a list of them as tuples 
    '''
    # We need to find all the ways the events overlap, then return a tuple of event ids 
    events_overlapped: list[tuple] = []
    events = sorted(events, key=lambda x: x[1]) # key must use a function for comparison

    for i, event in enumerate(events): 
        ind_overlaps = overlap_individual(event=event, i=i+1, total_events=events, sort=True)
        events_overlapped.extend(ind_overlaps)
        
    return events_overlapped

Naively it works, but the overlap count appears to be very low. This suggests that the individual birds in the sites typically do not interrupt one another.

Now, we should specify upper and lower bounds to increase the overlaps. Eg. Two events overlap if they occurred within x minutes of one another.

In [ ]:
cols = subst_loaded_events_df.columns
time_df = subst_loaded_events_df[['event_key', 'start_timestamp', 'end_timestamp']]

#### Collecting Metrics
Spatial: The distance between individual birds

Temporal: The time between individual bird interactions
- Will be user defined (30 meters from each other)

How do we calculate the distance of this bird from those around it?
- Option 1: For all birds in the same site, we can index it by id, then only update those birds
- Option 2: We can use ${\{site, (x, y)\}}$ as the key, and bird_id list as the value to drastically cut down on searching time

A slider will have to be added to denote a time range of focus. We can use the same overlap algorithm as we did prior.
- User selects the start timestamp and the end timestamp of focus

### Mathematical Model
In order to capture the interactions as precisely as possible, a mathematical model comprising the spatial & temporal aspects of the data will be formulated.

Here are the symbols that will be utilized for this formulations:
- $I$ denotes the interaction between two individual birds
- $I, J$ denote the observed acoustic events of two distinct individual birds
- $\beta_i \in \{\beta_0, \beta_1, \ldots \}$ denote the weights that should be applied
- $S(i, j)$ denotes the Euclidean Distance between i and j in meters, normalized to [0, 1] in which 1 is the closest
- $T(i, j)$ denotes the Temporal Proximity between i and j in minutes, normalized to [0, 1] in which 1 meaning no delay
- $H(i, j)$ denotes the number of historical interactions between i and j, normalized to [0, 1]
- $\theta_{ij} = \beta_0 + \beta_1 \times S(i, j) + \beta_2 \times T(i, j) + \beta_3 \times H(i, j)$ 
- $\sigma = \frac{1}{1 + e^{-\theta_{ij}}}$ denotes the logistic sigmoid function that maps into the range [0, 1]

$P(I \mid X, Y) = \sigma [\beta_0 + \beta_1 \times S(X, Y) + \beta_2 \times T(X, Y) + \beta_3 \times H(X, Y)] = \sigma [\theta_{ij}] = z$

In other words, the probability of an interaction occurring given evidence for X and Y is the logistic transformation of the linear combination of the Spatial Temporal Evidence. 

$\beta$ coefficients are free parameters that will be estimated from the data via Bayesian Inference

$P(I \mid X, Y) \propto P(X, Y \mid I) \times P(I)$

The above is the Bayesian formulation for the same phenomena. The probability that an interaction has occurred given evidence from two birds is equivalent to the probability of the evidence given the interaction times the probability of the interaction occurring.

In [ ]:
# event lower and upper bounds
lower_s = 5
upper_e = 5

# distance among birds
dist = 30 

Now, we will create a new dataframe that will store the scaled start and end timestamps.

In [ ]:
# create a new column that scales the start and end
time_df['scale_time_start'] = time_df['start_timestamp'] - pd.to_timedelta(lower_s, unit='minute')
time_df['scale_time_end'] = time_df['end_timestamp'] + pd.to_timedelta(upper_e, unit='minute')

# set the event_key as the index
time_df = time_df.set_index("event_key")

In [ ]:
list(time_df.columns)

Set the time range to be within the minimum and maximum timestamps to setup the time slider

In [ ]:
# Time Ranger Slider Configuration
slider_lower_s = min(time_df['scale_time_start'])
slider_upper_e = max(time_df['scale_time_end'])

At this point, we have sucessfully transformed those columns by adding and subtracting increments

In [ ]:
# Check this
tuples_events_scale = create_tuple_list_from_df(time_df, 'start_timestamp', 'end_timestamp')

Now that we have successfully transformed the original dataframe to reflect a broader range, we will run the overlap algorithm on this.

In [ ]:
# Check this
event_overlapping_ids = overlap(tuples_events_scale) # We now have a list of overlapping event ids

Test run the overlap algorithm for one individual

In [ ]:
tuples_events_scale[4]

In [ ]:
individual = tuples_events_scale[4]
ind_overlap = overlap_individual(individual, 0, tuples_events_scale, True)
len(ind_overlap)

To implement a time slider effect, we will create a new "event" with start and end timestamps. Then, we will allow the user to change where the window is located.

For instance, $\{{id, start, end\}}$ will be designated our time window. To slide across the events, we will add in a increment or decrement function. Then, we will run the overlap_individual function that will be used.


### Identifying Individuals
Using the unique event ids, we will find the corresponding individuals and plot only those events that occurred at the same time.

In [ ]:
def filter_events_individuals(id_list, time):
    '''
    Filters the event list and returns a list of tuples of individuals that overlap. \n
    If time is True, then overlap by that prior \n 
    '''
    individuals = [] # (individual_1, individual_2)
    for id in id_list:
        e_1 = subst_loaded_events_df.loc[id[0]]
        e_2 = subst_loaded_events_df.loc[id[1]]

        ind_event_1 = e_1["individual"]
        ind_event_2 = e_2["individual"]

        b_1 = int(ind_event_1)  # type: ignore
        b_2 = int(ind_event_2) # type: ignore

        # Adding the time filter here soon
        if e_1["array"] != e_2["array"] or b_1 == b_2: # Check that the site is the same and bird is unique
            continue 
        
        individuals.append( # append the bird & it's row info so we do not have to search again
            (
            {
                "name": b_1,
                "info": e_1
            },
            {
                "name": b_2,
                "info": e_2
            }
            )
        )
    return individuals
    

In [ ]:
individuals = filter_events_individuals(event_overlapping_ids, False)

In [ ]:
individual_filtered_list = filter_events_individuals(ind_overlap, False)

### Graph Implementation

Now we have a list of the individuals with overlapping acoustic records, we have enough information to create visualization of the graphs.

Import the networkX library in order to store the individuals into the graph.

In [ ]:
import networkx as nx

In [ ]:
# Now we can iterate over the list of tuples, adding them to the Graph
# ({name, info}, {name, info}) : These two individual birds had an event overlap, each with its own metadata
bird_net = nx.Graph()
#individuals

for overlap in individuals:
    a = overlap[0]["name"]
    b = overlap[1]["name"]
    
    bird_net.add_node(a)
    bird_net.add_node(b)

    if bird_net.has_edge(a, b):
        bird_net[a][b]["weight"] += 1  # increment on repeat overlap
    else:
        bird_net.add_edge(a, b, weight=1)

### Graph Visualization
Now, we will use Pyvis to visualize the bird interaction network. Since the nodes and edges tend to cluster up and overlap when using matplotlib, Pyvis was chosen due to its repulsion physics that make it more understandable.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from pyvis.network import Network

components = list(nx.connected_components(bird_net))
cmap = plt.get_cmap("tab20") # 20 distinct colors

In [ ]:
# Build a site -> color mapping
site_map = {}  # node -> site
for node in bird_net.nodes(): 
    site_map[node] = subst_loaded_events_df[
        subst_loaded_events_df["individual"] == node
    ].iloc[0]["array"]

sites = list(set(site_map.values())) # names of the arrays
indices = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 1]  # 11 distinct indices from tab20
site_colors = {site: cmap(indices[i]) for i, site in enumerate(sites)} # maps each site to color

In [ ]:
net = Network(notebook=True, height="750px", width="100%")

# Add nodes
for i, component in enumerate(components): # Color changes based on components
    for node in component: # Will be the same color
        site = site_map[node]
        color = site_colors[site] # fetch the color for each component
        hex_color = "#{:02x}{:02x}{:02x}".format(
            int(color[0]*255), int(color[1]*255), int(color[2]*255)
        )
        net.add_node(node,
                     label=str(node),
                     color=hex_color,
                     font={"size": 14, "color": "black", "align": "center"})

# Add edges with normalized weights
weights = [data.get("weight", 1) for _, _, data in bird_net.edges(data=True)]
min_w, max_w = min(weights), max(weights)

for src, dest, data in bird_net.edges(data=True):
    w = data.get("weight", 1)
    normalized = 1 + 14 * (w - min_w) / (max_w - min_w)
    net.add_edge(src, dest, value=normalized)

# Toggle the buttons and show the canvas
net.show_buttons(filter_=["physics"])
net.show("bird_net.html")

### Larger Data Set
Prior, we worked with the small Localization data set to gain familiarity. In the following section, we will extend these methods to the larger unannotated data set.

The goal is to create a network from the much larger data set and examine how well the methodology used above generalizes to this. The Snowy HPC at the Kitzes Lab will be the primary medium that will be used to execute the following scripts.